# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardikkk-1209/ML_Pipeline/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames the **Refresh / Content Opportunity Scoring** lane. Work the sections in order — simple words, honest numbers.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [ ]:
# Check-only cell.
print("Lane: Refresh / Content Opportunity Scoring")
print("Task: classification with probability scoring used to rank pages for review.")

## 1. My lane as an ML task

### Lane
**Refresh / Content Opportunity Scoring**

### Task type
**Classification with probability scoring for ranking.**

The practical decision is which existing content pages a content or SEO team should review first for a possible refresh. The model can estimate the probability that a page is in the declining/opportunity group, and that probability can be used to order a review queue.

This is decision support, not an automatic instruction to refresh a page.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [ ]:
# The starter dataset exposes a current-window proxy.
# The full-release work can later replace it with a genuinely future-looking label.
print("Starter proxy: is_declining_label = (trend_direction == 'down')")
print("Final capstone target: an outcome measured in a later observation window.")

## 2. Target or proxy

### Target / proxy

For the **starter workflow**, the proxy is `is_declining_label`, defined as `trend_direction == "down"`. It is useful for building and checking the workflow, but it is not a future outcome.

For the stronger capstone evaluation, the target should be constructed from the daily warehouse: use historical features before a decision time and measure performance movement in a later window. That prevents the model from simply learning a label already present in the same snapshot.

I will not use `trend_direction` or `trend_pct` as model features because they are used to define the starter proxy.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
print("Primary metric: Precision@10% of the ranked review queue.")
print("Comparison: beat or improve on the fixed-rule baseline on the same held-out data.")

## 3. Success metric

The primary metric is **Precision@10%** of the ranked review queue.

It answers: among the top 10% of pages sent to reviewers, what share belong to the decline/opportunity outcome? A useful model should beat the fixed-rule baseline on the same held-out data.

I will also report the outcome base rate and recall at the same K so the result is not interpreted without context.

## 4. The unit of analysis, as a real dataframe

*Load the starter lane slice and show it: one row = one what?*

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns):,}")
print("One row represents one pseudonymized content page.")
df.head()

## 4. The unit of analysis

The starter dataset uses **one row per content page**. Each row contains page-level search, traffic, content, and performance signals for that page.

The identifiers `content_id` and `client_id` are used for identification and grouping, not as model features.

In [ ]:
# Verify the unit of analysis and inspect the fields used by this lane.
print("Unique content IDs:", df["content_id"].nunique())
print("Duplicate content IDs:", df["content_id"].duplicated().sum())

lane_fields = [
    "impressions_90d", "clicks_90d", "sessions_90d", "ctr",
    "avg_position", "content_age_days", "days_since_last_update",
    "engagement_rate", "scroll_rate", "trend_direction"
]
print("\nAvailable lane fields:")
print([c for c in lane_fields if c in df.columns])

## 5. Why ML beats a fixed rule here

A fixed rule could say “refresh pages older than X days with more than Y impressions.” That is easy to explain, but it forces us to choose thresholds by hand and may ignore useful combinations of visibility, clicks, CTR, position, traffic, content age, and engagement.

A data-driven model can combine several observable signals and produce a comparable priority score. The model still needs to beat the simple baseline on an honest evaluation split before it earns a place in the workflow.

The final output remains **decision support for a human reviewer**.

## Self-check

- [x] Lane is Refresh / Content Opportunity Scoring.
- [x] Task type and decision are explicit.
- [x] Starter target is clearly labelled as a proxy, not a future causal outcome.
- [x] `Precision@10%` is named before model evaluation.
- [x] IDs and label-derived fields are excluded from features.
- [x] The notebook loads and verifies the real starter dataframe.
- [x] Claims use careful, decision-support language.